# Baseline oficial

Este notebook avalia o baseline oficial da tarefa de classificação de
clareza:

- representação TF-IDF;
- Regressão Logística;
- `class_weight="balanced"`.

A avaliação utiliza os cinco folds previamente definidos com
`StratifiedGroupKFold`.

O vetorizador TF-IDF é ajustado independentemente em cada conjunto de
treinamento, evitando vazamento de informação para a validação.

In [18]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [19]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [20]:
from src.evaluation.folds import carregar_folds
from src.models.baseline import criar_baseline_oficial

In [21]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "train.xlsx"
FOLDS_PATH = PROJECT_ROOT / "data" / "splits" / "folds.csv"

df = pd.read_excel(
    DATA_PATH,
    sheet_name="train",
)

df["resp_text"] = df["resp_text"].astype(str)

df = carregar_folds(
    df,
    FOLDS_PATH,
)

df.head()

,resp_text,clarity,fold
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5,3
1,"Prezada cidadã, As informações sobre óbitos ...",c1,3
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1,4
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234,4
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234,0


In [22]:
resultados = []

predicoes_oof = np.empty(
    len(df),
    dtype=object,
)

for fold in sorted(df["fold"].unique()):
    print(f"\n===== Fold {fold} =====")

    treino = df[df["fold"] != fold]
    validacao = df[df["fold"] == fold]

    modelo = criar_baseline_oficial()

    inicio = time.perf_counter()

    modelo.fit(
        treino["resp_text"],
        treino["clarity"],
    )

    predicoes = modelo.predict(
        validacao["resp_text"]
    )

    tempo = time.perf_counter() - inicio

    accuracy = accuracy_score(
        validacao["clarity"],
        predicoes,
    )

    f1_macro = f1_score(
        validacao["clarity"],
        predicoes,
        average="macro",
    )

    predicoes_oof[validacao.index] = predicoes

    resultados.append({
        "fold": fold,
        "n_treino": len(treino),
        "n_validacao": len(validacao),
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "tempo_segundos": tempo,
    })

    print(f"Treino: {len(treino)}")
    print(f"Validação: {len(validacao)}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro-F1: {f1_macro:.4f}")
    print(f"Tempo: {tempo:.2f}s")


===== Fold 0 =====
Treino: 16073
Validação: 4019
Accuracy: 0.4474
Macro-F1: 0.4447
Tempo: 3.50s

===== Fold 1 =====


c:\Users\valer\Projects\PLN\Processamento-de-Linguagem-Natural\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Treino: 16073
Validação: 4019
Accuracy: 0.4586
Macro-F1: 0.4572
Tempo: 3.55s

===== Fold 2 =====
Treino: 16073
Validação: 4019
Accuracy: 0.4436
Macro-F1: 0.4412
Tempo: 3.25s

===== Fold 3 =====
Treino: 16076
Validação: 4016
Accuracy: 0.4509
Macro-F1: 0.4494
Tempo: 3.32s

===== Fold 4 =====


c:\Users\valer\Projects\PLN\Processamento-de-Linguagem-Natural\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Treino: 16073
Validação: 4019
Accuracy: 0.4583
Macro-F1: 0.4556
Tempo: 3.52s


In [23]:
resultados_df = pd.DataFrame(resultados)

resultados_df

,fold,n_treino,n_validacao,accuracy,f1_macro,tempo_segundos
0,0,16073,4019,0.447375,0.444703,3.503807
1,1,16073,4019,0.458572,0.457162,3.552267
2,2,16073,4019,0.443643,0.441190,3.249435
3,3,16076,4016,0.450946,0.449395,3.321464
4,4,16073,4019,0.458323,0.455594,3.520044


In [24]:
accuracy_media = resultados_df["accuracy"].mean()
accuracy_std = resultados_df["accuracy"].std()

f1_media = resultados_df["f1_macro"].mean()
f1_std = resultados_df["f1_macro"].std()

print(
    f"Accuracy: "
    f"{accuracy_media:.4f} ± {accuracy_std:.4f}"
)

print(
    f"Macro-F1: "
    f"{f1_media:.4f} ± {f1_std:.4f}"
)

Accuracy: 0.4518 ± 0.0066
Macro-F1: 0.4496 ± 0.0069


In [25]:
accuracy_oof = accuracy_score(
    df["clarity"],
    predicoes_oof,
)

f1_oof = f1_score(
    df["clarity"],
    predicoes_oof,
    average="macro",
)

print(f"Accuracy OOF global: {accuracy_oof:.4f}")
print(f"Macro-F1 OOF global: {f1_oof:.4f}")

Accuracy OOF global: 0.4518
Macro-F1 OOF global: 0.4496


In [26]:
labels = ["c1", "c234", "c5"]

matriz = confusion_matrix(
    df["clarity"],
    predicoes_oof,
    labels=labels,
)

matriz_df = pd.DataFrame(
    matriz,
    index=[f"Real {label}" for label in labels],
    columns=[f"Predito {label}" for label in labels],
)

matriz_df

,Predito c1,Predito c234,Predito c5
Real c1,3214,1685,1448
Real c234,2243,2349,2261
Real c5,1527,1851,3514


In [27]:
print(
    classification_report(
        df["clarity"],
        predicoes_oof,
        labels=labels,
        digits=4,
    )
)

              precision    recall  f1-score   support

          c1     0.4602    0.5064    0.4822      6347
        c234     0.3992    0.3428    0.3688      6853
          c5     0.4865    0.5099    0.4979      6892

    accuracy                         0.4518     20092
   macro avg     0.4486    0.4530    0.4496     20092
weighted avg     0.4484    0.4518    0.4489     20092



In [28]:
RESULTS_DIR = PROJECT_ROOT / "results"

oof_df = pd.DataFrame({
    "row_index": df.index,
    "classe_real": df["clarity"],
    "predicao": predicoes_oof,
    "fold": df["fold"],
})

oof_df.to_csv(
    RESULTS_DIR / "baseline_oof.csv",
    index=False,
)

resultados_df.to_csv(
    RESULTS_DIR / "baseline_folds.csv",
    index=False,
)

In [29]:
pd.read_csv(
    RESULTS_DIR / "baseline_folds.csv"
)

,fold,n_treino,n_validacao,accuracy,f1_macro,tempo_segundos
0,0,16073,4019,0.447375,0.444703,3.503807
1,1,16073,4019,0.458572,0.457162,3.552267
2,2,16073,4019,0.443643,0.441190,3.249435
3,3,16076,4016,0.450946,0.449395,3.321464
4,4,16073,4019,0.458323,0.455594,3.520044


In [30]:
pd.read_csv(
    RESULTS_DIR / "baseline_oof.csv"
).head()

,row_index,classe_real,predicao,fold
0,0,c5,c1,3
1,1,c1,c1,3
2,2,c1,c1,4
3,3,c234,c1,4
4,4,c234,c5,0


In [31]:
experimento_baseline = pd.DataFrame([{
    "experimento": "baseline_oficial",
    "modelo": "TF-IDF + Logistic Regression",
    "accuracy_media": accuracy_media,
    "accuracy_std": accuracy_std,
    "f1_macro_media": f1_media,
    "f1_macro_std": f1_std,
}])

experimento_baseline.to_csv(
    RESULTS_DIR / "experiments.csv",
    index=False,
)

experimento_baseline

,experimento,modelo,accuracy_media,accuracy_std,f1_macro_media,f1_macro_std
0,baseline_oficial,TF-IDF + Logistic Regression,0.451772,0.006619,0.449609,0.006853
